## TASK 8. METADATA FILTERING
### Objective

This task demonstrates how metadata can be attached to vectors and used to filter search results.

Goals : 
- Add metadata to vector embeddings.
- Store metadata in ChromaDB.
- Perform filtered semantic search.
- Compare search results using different filters.

## Notebook Structure

1.Objective
2.Install Libraries
3.Import Libraries.
4.Load API Key
5.Create Documents with Metadata
6.Create Chroma Collection
7.Generate Embeddings
8.Store Documents and Metadata
9.Perform Metadata Filtering
10.Analysis
11.Conclusion

## Step 1: Install Libraries

In [1]:
!pip install chromadb openai python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Import Libraries

In [2]:
import os

from dotenv import load_dotenv
from openai import OpenAI

import chromadb

## Step 3: Load API Key

In [3]:

load_dotenv()

# Get the OpenAI API keys from environment variables
openai_api_key = os.getenv("API_KEY")

# Let's configure the OpenAI Client using our key
openai_client = OpenAI(api_key = openai_api_key)
print("\nOpenAI client successfully configured.")

# Let's view the first few characters in the key
print(openai_api_key[:3])


OpenAI client successfully configured.
sk-


## Step 4: Create Documents with Metadata

In [12]:
documents = [
    "Machine learning learns patterns from data.",
    "Deep learning uses neural networks.",
    "SQL databases store structured data.",
    "MongoDB is a NoSQL database.",
    "Football analytics improves team performance.",
    "Cricket is popular in India.",
    "Electric vehicles reduce pollution.",
    "Solar energy supports sustainability."
]

metadata = [
    {"category": "AI", "level": "Beginner", "author": "John"},
    {"category": "AI", "level": "Advanced", "author": "John"},
    {"category": "Database", "level": "Beginner", "author": "David"},
    {"category": "Database", "level": "Advanced", "author": "David"},
    {"category": "Sports", "level": "Advanced", "author": "Mike"},
    {"category": "Sports", "level": "Beginner", "author": "Mike"},
    {"category": "Environment", "level": "Beginner", "author": "Emma"},
    {"category": "Environment", "level": "Advanced", "author": "Emma"}
]

In [ ]:
df = pd.DataFrame(metadata)
df["Document"] = documents

df

,category,level,author,Document
0,AI,Beginner,John,Machine learning learns patterns from data.
1,AI,Advanced,John,Deep learning uses neural networks.
2,Database,Beginner,David,SQL databases store structured data.
3,Database,Advanced,David,MongoDB is a NoSQL database.
4,Sports,Advanced,Mike,Football analytics improves team performance.
5,Sports,Beginner,Mike,Cricket is popular in India.
6,Environment,Beginner,Emma,Electric vehicles reduce pollution.
7,Environment,Advanced,Emma,Solar energy supports sustainability.


## Step 5: Create Chroma Collection

In [5]:
client = chromadb.Client()

collection = client.create_collection(
    name="metadata_collection"
)

print("Collection created.")

Collection created.


## Step 6: Generate Embeddings and Store Data

In [6]:
for i, doc in enumerate(documents):

    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=doc
    )

    embedding = response.data[0].embedding

    collection.add(
        ids=[str(i)],
        documents=[doc],
        embeddings=[embedding],
        metadatas=[metadata[i]]
    )

print("Documents and metadata stored.")

Documents and metadata stored.


## Step 7: Metadata Filter 1 (Category = AI)

In [7]:
query = "How do computers learn?"

query_embedding = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"category": "AI"}
)

print(results["documents"])

[['Machine learning learns patterns from data.', 'Deep learning uses neural networks.']]


## Step 8: Metadata Filter 2 (Category = Database)

In [8]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"category": "Database"}
)

print(results["documents"])

[['SQL databases store structured data.', 'MongoDB is a NoSQL database.']]


## Step 9: Metadata Filter 3 (Beginner Level)

In [9]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"level": "Beginner"}
)

print(results["documents"])

[['Machine learning learns patterns from data.', 'SQL databases store structured data.', 'Electric vehicles reduce pollution.', 'Cricket is popular in India.']]


## Step 10: Combined Metadata Filter

In [11]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={
        "$and": [
            {"category": "AI"},
            {"level": "Beginner"}
        ]
    }
)
print("Query:", query)
print("Category: AI")
print(results["documents"])

Query: How do computers learn?
Category: AI
[['Machine learning learns patterns from data.']]


## Analysis

Metadata filtering allows semantic search systems to combine vector similarity with structured information.

Observations:

- The AI filter returned only artificial intelligence documents.
- The Database filter returned database-related documents.
- The Beginner filter returned introductory-level content.
- The combined filter returned highly specific results.

Metadata improves retrieval quality by reducing irrelevant search results and enabling targeted information retrieval.

This approach is commonly used in RAG systems, enterprise search applications, and educational assistants.

## Conclusion

Metadata filtering enhances semantic search by combining vector similarity with structured attributes.

Vector databases such as ChromaDB support filtering using categories, difficulty levels, dates, authors, and other metadata fields.

Metadata filtering is widely used in:

- RAG applications
- Enterprise search
- Educational chatbots
- Recommendation systems
- Knowledge management systems

This experiment demonstrates how semantic search and metadata work together to improve retrieval accuracy.